# Upload biLlama to HuggingFace Hub

Uploads **only code + config** — no weights.
Users load Llama weights separately from the original gated repo.

Target: `meta-llama/Llama-3.1-8B-Instruct` → `{HUB_REPO}`

Architecture changes:
- Identical weight structure to Llama (all keys match 1:1)
- Adds `use_bidirectional_attention` flag to config
- When enabled: replaces causal mask with full attention mask at forward pass level

In [1]:
import json
from pathlib import Path
from huggingface_hub import HfApi
from gpt2vec.models.bi_llama import (
    biLlamaConfig,
    biLlamaModel,
    biLlamaForCausalLM,
    biLlamaForTokenClassification,
)

## Configuration

Set your HuggingFace username and desired repo name.

In [2]:
HUB_REPO   = "nuinashco/biLlama-3.1-8B-Instruct"  # <-- fill in
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

# Local directory to stage files before upload
LOCAL_DIR  = Path("../tmp/bi_llama_upload")

# Path to source code files
BI_LLAMA_SRC = Path("../src/gpt2vec/models/bi_llama")

## Step 1 — Register config class with HF Auto system

`biLlamaConfig.register_for_auto_class()` is needed so `save_pretrained()` calls
`custom_object_save`, which copies the config source file and writes the `AutoConfig` entry
into `config.json`.

**Model class entries are NOT written automatically** — `register_for_auto_class()` on model
classes only sets `_auto_class` on them; it does not propagate into the config's `auto_map`.
We set model entries manually in Step 3.

In [3]:
biLlamaConfig.register_for_auto_class()  # triggers custom_object_save on save_pretrained

## Step 2 — Build biLlamaConfig from Llama-3.1-8B-Instruct

Copies all Llama hyperparameters and overrides `model_type` + adds `use_bidirectional_attention`.

In [4]:
config = biLlamaConfig.from_pretrained(
    BASE_MODEL,
    use_bidirectional_attention=True,
)
print(f"model_type              : {config.model_type}")
print(f"use_bidirectional_attn  : {config.use_bidirectional_attention}")
print(f"hidden_size             : {config.hidden_size}")
print(f"num_hidden_layers       : {config.num_hidden_layers}")
print(f"num_attention_heads     : {config.num_attention_heads}")
print(f"vocab_size              : {config.vocab_size}")

You are using a model of type llama to instantiate a model of type bi_llama. This is not supported for all configurations of models and can yield errors.


model_type              : llama
use_bidirectional_attn  : True
hidden_size             : 4096
num_hidden_layers       : 32
num_attention_heads     : 32
vocab_size              : 128256


## Step 3 — Set `auto_map` and save config locally

Two issues with the naive approach:
1. `save_pretrained` only writes the config class's own `AutoConfig` entry — model entries
   must be added manually.
2. `AutoModelForCausalLM.from_pretrained(BASE_MODEL, config=cfg)` uses `BASE_MODEL` as the
   repo to fetch remote code from. This fails because `modeling_bi_llama.py` lives in
   `HUB_REPO`, not in the Llama repo.

Fix: prefix model entries with `"HUB_REPO--"`. `get_class_from_dynamic_module` splits on
`"--"` and downloads code from the left-hand repo regardless of where weights come from.

In [5]:
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# Manually build the complete auto_map.
# Model entries use "HUB_REPO--module.Class" so AutoModel resolves code from HUB_REPO
# even when weights are loaded from a different repo (BASE_MODEL).
config.auto_map = {
    "AutoConfig":                       "configuration_bi_llama.biLlamaConfig",
    "AutoModel":                        f"{HUB_REPO}--modeling_bi_llama.biLlamaModel",
    "AutoModelForCausalLM":             f"{HUB_REPO}--modeling_bi_llama.biLlamaForCausalLM",
    "AutoModelForTokenClassification":  f"{HUB_REPO}--modeling_bi_llama.biLlamaForTokenClassification",
}

config.save_pretrained(LOCAL_DIR)

# Verify auto_map was written correctly
cfg_json = json.loads((LOCAL_DIR / "config.json").read_text())
print("auto_map in config.json:")
print(json.dumps(cfg_json.get("auto_map", {}), indent=2))

auto_map in config.json:
{
  "AutoConfig": "configuration_bi_llama.biLlamaConfig",
  "AutoModel": "nuinashco/biLlama-3.1-8B-Instruct--modeling_bi_llama.biLlamaModel",
  "AutoModelForCausalLM": "nuinashco/biLlama-3.1-8B-Instruct--modeling_bi_llama.biLlamaForCausalLM",
  "AutoModelForTokenClassification": "nuinashco/biLlama-3.1-8B-Instruct--modeling_bi_llama.biLlamaForTokenClassification"
}


## Step 4 — Create Hub repo and upload

Uploads:
- `configuration_bi_llama.py`  (custom config class)
- `modeling_bi_llama.py`       (custom model classes)
- `config.json`                (biLlama hyperparams + correct `auto_map`)

**No weights uploaded.** Users point `from_pretrained` at the original Llama repo for weights.

In [6]:
api = HfApi()
api.create_repo(HUB_REPO, repo_type="model", exist_ok=True)
print(f"Repo ready: https://huggingface.co/{HUB_REPO}")

Repo ready: https://huggingface.co/nuinashco/biLlama-3.1-8B-Instruct


In [7]:
# Upload code files
for filename in ["configuration_bi_llama.py", "modeling_bi_llama.py"]:
    api.upload_file(
        path_or_fileobj=str(BI_LLAMA_SRC / filename),
        path_in_repo=filename,
        repo_id=HUB_REPO,
    )
    print(f"✓ {filename}")

# Upload config
api.upload_file(
    path_or_fileobj=str(LOCAL_DIR / "config.json"),
    path_in_repo="config.json",
    repo_id=HUB_REPO,
)
print("✓ config.json")

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


✓ configuration_bi_llama.py
✓ modeling_bi_llama.py
✓ config.json


## Step 5 — Verify: load from Hub

### Option A — `gpt2vec` package installed (no `trust_remote_code` needed)

```python
from gpt2vec.models.bi_llama import biLlamaConfig, biLlamaForCausalLM

config = biLlamaConfig.from_pretrained(HUB_REPO)
model  = biLlamaForCausalLM.from_pretrained(
    BASE_MODEL,    # weights from original gated Llama repo
    config=config, # architecture from your hub repo
    torch_dtype="auto",
    device_map="auto",
)
```

### Option B — pure `transformers`, `trust_remote_code=True` (no install needed)

`auto_map` entries in config.json use `"HUB_REPO--module.Class"` format.
`AutoModelForCausalLM.from_pretrained` sees `"--"`, splits it, and fetches
code from `HUB_REPO` while loading weights from `BASE_MODEL`.

```python
from transformers import AutoConfig, AutoModelForCausalLM

cfg   = AutoConfig.from_pretrained(HUB_REPO, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    config=cfg,
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto",
)
```

In [8]:
# Smoke test: verify config + auto_map round-trip correctly from Hub
from transformers import AutoConfig

loaded_cfg = AutoConfig.from_pretrained(HUB_REPO, trust_remote_code=True)

assert type(loaded_cfg).__name__ == "biLlamaConfig", f"Expected biLlamaConfig, got {type(loaded_cfg)}"
assert loaded_cfg.use_bidirectional_attention is True
assert loaded_cfg.model_type == "bi_llama"
assert loaded_cfg.hidden_size == config.hidden_size
assert "AutoModelForCausalLM" in loaded_cfg.auto_map, "auto_map missing AutoModelForCausalLM"
assert HUB_REPO in loaded_cfg.auto_map["AutoModelForCausalLM"], "auto_map missing HUB_REPO-- prefix"

print(f"Config loaded successfully from {HUB_REPO}")
print(f"  type                    : {type(loaded_cfg).__name__}")
print(f"  model_type              : {loaded_cfg.model_type}")
print(f"  use_bidirectional_attn  : {loaded_cfg.use_bidirectional_attention}")
print(f"  hidden_size             : {loaded_cfg.hidden_size}")
print(f"  auto_map                : {loaded_cfg.auto_map}")

config.json: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/nuinashco/biLlama-3.1-8B-Instruct:
- configuration_bi_llama.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Config loaded successfully from nuinashco/biLlama-3.1-8B-Instruct
  type                    : biLlamaConfig
  model_type              : bi_llama
  use_bidirectional_attn  : True
  hidden_size             : 4096
  auto_map                : {'AutoConfig': 'configuration_bi_llama.biLlamaConfig', 'AutoModel': 'nuinashco/biLlama-3.1-8B-Instruct--modeling_bi_llama.biLlamaModel', 'AutoModelForCausalLM': 'nuinashco/biLlama-3.1-8B-Instruct--modeling_bi_llama.biLlamaForCausalLM', 'AutoModelForTokenClassification': 'nuinashco/biLlama-3.1-8B-Instruct--modeling_bi_llama.biLlamaForTokenClassification'}


## Weight compatibility check (optional)

Verifies that biLlama loads Llama-3.1-8B-Instruct weights with zero missing/unexpected keys.

`auto_map["AutoModelForCausalLM"]` is `"HUB_REPO--modeling_bi_llama.biLlamaForCausalLM"`.
`from_pretrained` splits on `"--"`, downloads code from `HUB_REPO`, loads weights from `BASE_MODEL`.

In [9]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,           # source of weights (Llama-3.1-8B-Instruct)
    config=loaded_cfg,    # biLlamaConfig with auto_map pointing to HUB_REPO
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(type(model).__name__)                      # biLlamaForCausalLM
print(model.config.use_bidirectional_attention)  # True

modeling_bi_llama.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/nuinashco/biLlama-3.1-8B-Instruct:
- modeling_bi_llama.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk.


biLlamaForCausalLM
True
